## Cell 1: Install & check GPU

In [1]:
!pip install -q transformers peft bitsandbytes accelerate

import torch
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Total VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:00:00:0100:01
Free VRAM: 15.53 GB
Total VRAM: 15.64 GB
Sun Mar  8 21:50:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             27W /   70W |     105MiB /  15360MiB |      5%      Default |
|        

In [2]:
import os

# Check adapter path
adapter_path = "/kaggle/input/models/somendrew/gen-z-slang-model/pytorch/default/1/content/genz-slang-model/checkpoint-48"

print("\nAdapter files:")
print(os.listdir(adapter_path))


Adapter files:
['adapter_model.safetensors', 'trainer_state.json', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'scheduler.pt', 'optimizer.pt', 'rng_state.pth']


## Cell 2: Uploading Base Model & Adapter

In [3]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

# Pull base model from HuggingFace directly
base_model_id  = "mistralai/Mistral-7B-Instruct-v0.2"

#  adapter is in the checkpoint-48 folder
adapter_path = adapter_path

print("Loading base model from HuggingFace...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,                 
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Loading adapter...")
peft_model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    torch_dtype=torch.bfloat16,
)

Loading base model from HuggingFace...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loading adapter...


## Cell 3: Merge

In [4]:
print("Merging...")
merged_model = peft_model.merge_and_unload()

del peft_model
gc.collect()
torch.cuda.empty_cache()

Merging...


## Cell 4: Save Model

In [5]:
print("Saving...")
merged_model.save_pretrained(
    "/kaggle/working/genz-merged-model",
    safe_serialization=True,
    max_shard_size="2GB"
)

Saving...


Writing model shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained("/kaggle/working/genz-merged-model")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

('/kaggle/working/genz-merged-model/tokenizer_config.json',
 '/kaggle/working/genz-merged-model/chat_template.jinja',
 '/kaggle/working/genz-merged-model/tokenizer.json')